### Feature Engineering 1

In [1]:
from util.data_aquisition import generate
from util.data_frame import get_data_frame
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = generate(seasons=[2015, 2016, 2017, 2018], play_offs=[False])

Data is available on your system. Will be loaded from: C:\Users\christian\Desktop\IFT-6758\data\regular_season\2015-2016
Data is available on your system. Will be loaded from: C:\Users\christian\Desktop\IFT-6758\data\regular_season\2016-2017
Data is available on your system. Will be loaded from: C:\Users\christian\Desktop\IFT-6758\data\regular_season\2017-2018


In [4]:
df = get_data_frame(data)
df.head()

,Game_ID,Game_Type,Event_ID,Home_Team,Home_Rink,Away_Team,Event_Type,Shoot_Type,Time,Period,Period_Time,Team_Attack,Power_Game,Empty_Net,Shooters_scorer,Goalies,X_Coordinate,Y_Coordinate
0,2015020001,R,7,Toronto Maple Leafs,right,Montréal Canadiens,Shot,Wrist Shot,2015-10-07T23:24:23Z,1,00:51,Toronto Maple Leafs,NaN,NaN,Brad Boyes,Carey Price,-55.0,6.0
1,2015020001,R,9,Toronto Maple Leafs,right,Montréal Canadiens,Shot,Snap Shot,2015-10-07T23:24:38Z,1,01:05,Toronto Maple Leafs,NaN,NaN,Nazem Kadri,Carey Price,-79.0,0.0
2,2015020001,R,10,Toronto Maple Leafs,right,Montréal Canadiens,Shot,Snap Shot,2015-10-07T23:27:21Z,1,01:06,Toronto Maple Leafs,NaN,NaN,Nazem Kadri,Carey Price,-75.0,-1.0
3,2015020001,R,13,Toronto Maple Leafs,right,Montréal Canadiens,Shot,Wrist Shot,2015-10-07T23:25:24Z,1,01:53,Toronto Maple Leafs,NaN,NaN,Daniel Winnik,Carey Price,-55.0,-36.0
4,2015020001,R,21,Toronto Maple Leafs,right,Montréal Canadiens,Shot,Tip-In,2015-10-07T23:27:08Z,1,02:37,Toronto Maple Leafs,NaN,NaN,Joffrey Lupul,Carey Price,-61.0,-1.0


#### Adding new features

In [ ]:
def transform_X(row):
    '''
    Given a row of the original data frame, this function transforms the coordinate X depending on the position of
    the teams in the rink. Since at the end the data is presented in a half-rink (we arbitrarily choose that
    this is the right half of the rink), when the attacking team plays on the right side of the rink , i.e
    the opponent's goal is on the left side, we need to flip the x coordinate.
    '''
    if str(row['Home_Rink']) == 'nan': #In case the rinkside information is not available, we assume the shot
        #is taken from the side that is the closest to the opponent's goal.
        return np.absolute(row['X_Coordinate'])

    else: #Otherwise we use rinkside information to determine whether to flip X coordinate as described above.
        if row['Home_Team'] == row['Team_Attack']:
            if row['Home_Rink'] == 'right':
                return -(row['X_Coordinate'])
            else:
                return row['X_Coordinate']
        else:
            if row['Home_Rink'] == 'left':
                return -(row['X_Coordinate'])
            else:
                return row['X_Coordinate']

In [ ]:
df['X_Transformed'] = df.apply(transform_X, axis=1)
df = df.assign(Distance_From_Net = lambda x : np.sqrt(pow(x["X_Transformed"] - 89, 2) + pow(x["Y_Coordinate"], 2)))
df = df.assign(Angle = lambda x : np.arctan((x["Y_Coordinate"]/(89-x["X_Transformed"])))*180/math.pi)
df["Is_Goal"] = df.Event_Type.apply(lambda x : 1 if x=="Goal" else 0)
df["Empty_Net_Bool"] = df.Empty_Net.apply(lambda x : 1 if x==True else 0)

In [ ]:
sns.set_theme()

In [ ]:
#histogramme du nombre de tirs (buts et non-buts séparés), regroupées (binned) par distance
g = sns.histplot(df, x="Distance_From_Net", hue="Event_Type", multiple="dodge", hue_order=['Goal', 'Shot'], shrink=.9, binwidth=20)
g.set_xlim(0, 200)
g.set_xticks(range(0, 200,20))
plt.title("Nombre de tirs (buts et non-buts séparés), binned par distance")
plt.show(g)

In [ ]:
#Un histogramme de nombre de tirs (buts et non-buts séparés), binned par angle
g = sns.histplot(df, x="Angle", hue="Event_Type", multiple="dodge", shrink=.9, hue_order=['Goal', 'Shot'], binwidth=20)
g.set_xlim(-180, 180)
g.set_xticks(range(-180, 180,20))
plt.title("Nombre de tirs (buts et non-buts séparés), binned par angle")
plt.show(g)

In [ ]:
sns.jointplot(data=df, x="Distance_From_Net", y="Angle", kind="hist", bins=20)

In [ ]:
#This function takes a series of event_types (possible types: Shot or Goal) and returns the rate of Goals.
def calculate_rate(event_types):
    Number_of_goals = event_types.value_counts()["Goal"]
    Number_of_shoot = event_types.value_counts()["Shoot"]

    #buts / (#pas_de_buts + #buts)
    return Number_of_goals / (Number_of_goals + Number_of_shoot)

In [ ]:
WIDTH = 5 #Bin width, can be changed depending on final output

In [ ]:
#Creating the 'Distance_bin' representing the bin corresponding to the distance of the shot
df['Distance_bin']=df.Distance_From_Net.map(lambda x: int(np.floor(x/WIDTH)*WIDTH) if not math.isnan(x) else np.nan)

In [ ]:
#Creating the 'Angle_bin' representing the bin corresponding to the angle of the shot
df['Angle_bin']=df.Angle.map(lambda x: int(np.floor(x/WIDTH)*WIDTH) if not math.isnan(x) else np.nan)

In [ ]:
df = df.dropna(subset='Distance_bin') #We don't want NaNs

In [ ]:
df = df.dropna(subset='Angle_bin') #We don't want NaNs

In [ ]:
#Grouping by distance bin and calculating the aggregate function: Rate of goals among the group
groups = df.groupby(by='Distance_bin').agg(goal_rate=pd.NamedAgg(column="Event_Type", aggfunc= calculate_rate))
groups = pd.DataFrame({'Distance_bin': groups.index, 'Goal_rate': groups.goal_rate})
plt.figure(figsize=(14,6))
g = sns.barplot(groups, x='Distance_bin', y='Goal_rate', color='royalblue')
g.set_xticklabels([f"[{float(i.get_text())}, {float(i.get_text())+WIDTH})" for i in g.get_xticklabels()], rotation = 90, ha="right")
plt.title(f"Goal rate as a function of distance")
plt.show()

In [ ]:
#Grouping by distance bin and calculating the aggregate function: Rate of goals among the group
groups = df.groupby(by='Angle_bin').agg(goal_rate=pd.NamedAgg(column="Event_Type", aggfunc= calculate_rate))
groups = pd.DataFrame({'Angle_bin': groups.index, 'Goal_rate': groups.goal_rate})
plt.figure(figsize=(14,6))
g = sns.barplot(groups, x='Angle_bin', y='Goal_rate', color='royalblue')
g.set_xticklabels([f"[{float(i.get_text())}, {float(i.get_text())+WIDTH})" for i in g.get_xticklabels()], rotation = 90, ha="right")
plt.title(f"Goal rate as a function of angle")
plt.show()